# 07d · Weather: ERA5 ground and air-mass extras from ARCO-ERA5

Two more pieces from **ARCO-ERA5** on Google Cloud Storage, with no CDS queue:

| Table | One row per | Content |
|---|---|---|
| `era5_static` | chip | ERA5's own ground height (its boundary layer is measured from this smoothed ground, not from the real terrain), land share, high and low vegetation cover and type, soil type, sub-grid relief |
| `era5_extra` | chip and UTC hour | skin (ground surface) temperature, top soil moisture, albedo, snow depth, water vapour in the column, freezing level, convective rain, friction velocity |

The ground variables are also in ERA5-Land at 0.1° (notebook 07, slower); these at 0.25° arrive today. Same hours as 07b: each site's competition days and the day before, 5 to 21 h local solar time. Each month is saved as soon as it is read, so **if Colab disconnects, Run All again**.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
THREADS = 32         # global fields read at a time
EVERY = 1            # read every hour
SOLAR = "5-21"       # local solar hours kept for each site
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive, get the code

In [ ]:
import pandas as pd
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

auth.authenticate_user()
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
assert os.path.isfile(os.path.join(DB, "site_chips.parquet")), "Run notebook 06 first: database/site_chips.parquet is missing"
subprocess.run(["pip", "install", "-q", "gcsfs", "zarr", "netCDF4"], check=True)
print("Database in", DB)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)


def run(*cmd):
    """Run a script and show its output; stopping the cell also stops the script (otherwise it would
    keep running in the background)."""
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
        code = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Stopped.")
        raise  # also stops Run All
    if code != 0:  # stop Run All here instead of building tables from an unfinished step
        raise RuntimeError(f"{os.path.basename(cmd[0])} stopped with an error (exit code {code}); see the lines above")
    return code

## 3. Fields that do not change

Once per chip, a few seconds: `weather/era5_static.parquet`.

In [ ]:
run(f"{REPO}/scripts/era5_arco.py", DB, "--product", "static")

## 4. What there is to read

How many hours and global fields, and whether some hours are newer than ARCO (those come from the CDS). Nothing is read yet.

In [ ]:
run(f"{REPO}/scripts/era5_arco.py", DB, "--product", "extra", "--threads", str(THREADS), "--every", str(EVERY), "--solar", SOLAR, "--plan-only")

## 5. Read

One line per month, with its speed and the time left. Stopping this cell also stops the reading; months already saved are kept.

In [ ]:
run(f"{REPO}/scripts/era5_arco.py", DB, "--product", "extra", "--threads", str(THREADS), "--every", str(EVERY), "--solar", SOLAR)

## 6. Build the table

Merges the months into `weather/era5_extra.parquet`.

In [ ]:
run(f"{REPO}/scripts/era5.py", DB, "--convert-only", "--tables", "extra")

## 7. A look at the table

In [ ]:
w = pd.read_parquet(os.path.join(DB, "weather", "era5_extra.parquet"))
print(f"{len(w):,} rows, {w.chip_id.nunique():,} chips, {w.time.dt.normalize().nunique():,} days")
print(w.drop(columns=["chip_id", "time"]).describe().T[["mean", "min", "50%", "max"]].round(1))